In [41]:
NO_RECORDS = 100

In [42]:
from datetime import datetime, timezone
from json import JSONDecodeError
from logging.config import dictConfig
from typing import Any, Optional
import psycopg
from psycopg import errors as psycopg_errors
from psycopg.rows import dict_row
from config.logging_config import LOGGING_CONFIG
from config.postgres_config import PostgresConfig
import os
from fastapi import FastAPI, Query, HTTPException
import logging

from lxml import etree as ET
# API calls
import requests


# added for these experiments
from utils.enrichment_methods import get_doi, clean_doi, batch_enrich_subjects_from_openaire, merge_subjects_from_new_method

In [43]:
dictConfig(LOGGING_CONFIG)
logger = logging.getLogger(__name__)

In [44]:
user = os.getenv("POSTGRES_USER")
passwd = os.getenv("POSTGRES_PASSWORD")
host = os.getenv("POSTGRES_ADDRESS", "127.0.0.1")


if host == '':
    host = "127.0.0.1"
    user = os.getenv("POSTGRES_ADMIN")

connection_params = {
    "user" : user,
    "password" : passwd,
    "dbname" : "datasetdb",
    "port" : 5432,
    "host" : host
}

In [45]:
query_records = """
            -- Exploring subjects from record raw metadata

-- Every xpath defines a potential XML structure with corresponding subject tag placement.
-- If raw_metadata conforms to ANY of these xpaths, subjects will be concatenated <subject> tags. (removing array_to_string() will return arrays)
-- If raw_metadata doesn't conform to any path (and probably has no subject tags), subjects will be ''


WITH query AS (
    SELECT
        records.*,
        array_to_string(
                xpath('oai:record/oai:metadata/datacite:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
                    '{{datacite, http://datacite.org/schema/kernel-4},
                      {oai, http://www.openarchives.org/OAI/2.0/}}'
                )::text[] ||
				xpath('oai:record/oai:metadata/oaidc:oai_datacite/oaidc:payload/datacite:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
                    '{{datacite, http://datacite.org/schema/kernel-4},
                      {oai, http://www.openarchives.org/OAI/2.0/},
                      {oaidc, http://schema.datacite.org/oai/oai-1.1/}}'
                )::text[] ||
				xpath('oai:record/oai:metadata/oai:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
                    '{{datacite, http://datacite.org/schema/kernel-4},
                      {oai, http://www.openarchives.org/OAI/2.0/}}'
                )::text[] ||
				xpath('oai:record/oai:metadata/oad:oai_datacite/oad:payload/datacite:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
				-- note the 3 in datacite
                    '{{datacite, http://datacite.org/schema/kernel-3},
					  {oai, http://www.openarchives.org/OAI/2.0/},
     				  {oad, http://schema.datacite.org/oai/oai-1.0/}}'
                )::text[] ||
				-- same path as previous but uses kernel-4 namespace
				xpath('oai:record/oai:metadata/oad:oai_datacite/oad:payload/datacite:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
                    '{{datacite, http://datacite.org/schema/kernel-4},
					  {oai, http://www.openarchives.org/OAI/2.0/},
     				  {oad, http://schema.datacite.org/oai/oai-1.0/}}'
                )::text[]
				, ', '
        ) AS subjects
    FROM public.records
)
SELECT 
    id,
    title,
    raw_metadata,
    doi,
    subjects
FROM query
WHERE query.subjects = ''
-- for these tests, we also want the records to have valid DOIs
    AND doi IS NOT NULL
ORDER BY id
LIMIT %s
;
"""

In [46]:
try:
    with psycopg.connect(**connection_params, row_factory=dict_row) as conn:
        cur = conn.cursor()
        
        cur.execute(query_records, (NO_RECORDS,))
        
        results = cur.fetchall()
        print(results[0])

        record_ids = [str(row["id"]) for row in results]
        record_titles = [str(row["title"]) for row in  results]
        record_dois = [row["doi"] for row in  results]
        record_raw_metadata = [str(row["raw_metadata"]) for row in  results]


except Exception as e:
    logger.exception("Failed to fetch harvest runs")
    raise HTTPException(status_code=500, detail=str(e))

{'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/102734481', 'title': 'Water structure around surfactant micelles', 'raw_metadata': '<record xmlns="http://www.openarchives.org/OAI/2.0/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance">\n  <header>\n    <identifier>oai:icat.isis.stfc.ac.uk:inv/102734481</identifier>\n    <datestamp>2019-03-18T15:13:33.990Z</datestamp>\n    <setSpec>doiInvs</setSpec>\n  </header>\n  <metadata>\n    <oai_datacite xmlns="http://schema.datacite.org/oai/oai-1.1/" xsi:schemaLocation="http://schema.datacite.org/oai/oai-1.1/ http://schema.datacite.org/oai/oai-1.1/oai.xsd">\n      <schemaVersion>4.3</schemaVersion>\n      <datacentreSymbol>BL.ISIS</datacentreSymbol>\n      <payload>\n        <resource xmlns="http://datacite.org/schema/kernel-4" xsi:schemaLocation="http://datacite.org/schema/kernel-4 http://schema.datacite.org/meta/kernel-4.3/metadata.xsd">\n          <identifier identifierType="DOI">https://doi.org/10.5286/ISIS.E.RB620144-1</identifier>\n 

# DOI check through bigger repositories  (#1)

## OpenAIRE

In [47]:
openaire_enriched = batch_enrich_subjects_from_openaire(record_dois)
openaire_enriched[:5]

DEBUG:    2026-07-02 11:35:49 urllib3.connectionpool@1049 Starting new HTTPS connection (1): api.openaire.eu:443
DEBUG:    2026-07-02 11:35:50 urllib3.connectionpool@544 https://api.openaire.eu:443 "GET /graph/v2/researchProducts?pid=10.5286%2Fisis.e.rb620144-1%2C10.5286%2Fisis.e.rb1790051-7%2C10.5286%2Fisis.e.rb1830641-1%2C10.5286%2Fisis.e.rb1830641-2%2C10.5286%2Fisis.e.rb1790346-2%2C10.5286%2Fisis.e.rb1790346-3%2C10.5286%2Fisis.e.rb1790346-5%2C10.5286%2Fisis.e.rb1790346-4%2C10.5286%2Fisis.e.rb1790346-6%2C10.5286%2Fisis.e.rb1790346-7 HTTP/1.1" 200 None
DEBUG:    2026-07-02 11:35:50 urllib3.connectionpool@1049 Starting new HTTPS connection (1): api.openaire.eu:443
DEBUG:    2026-07-02 11:35:51 urllib3.connectionpool@544 https://api.openaire.eu:443 "GET /graph/v2/researchProducts?pid=10.5286%2Fisis.e.rb1670547-3%2C10.5286%2Fisis.e.rb1990292-1%2C10.5286%2Fisis.e.rb1890354-2%2C10.5286%2Fisis.e.rb1510265-2%2C10.5286%2Fisis.e.rb1920617-3%2C10.5286%2Fisis.e.rb1790175-3%2C10.5286%2Fisis.e.rb1

[{'doi': 'https://doi.org/10.5286/ISIS.E.RB620144-1',
  'enriched_subjects': set()},
 {'doi': '10.5286/isis.e.rb1790051-7',
  'enriched_subjects': {'Construction Engineering and Architecture',
   'Engineering',
   'Engineering Sciences',
   'Materials',
   'STFC ISIS Neutron and Muon Data'}},
 {'doi': '10.5286/isis.e.rb1830641-1',
  'enriched_subjects': {'STFC ISIS Neutron and Muon Data'}},
 {'doi': '10.5286/isis.e.rb1830641-2',
  'enriched_subjects': {'STFC ISIS Neutron and Muon Data'}},
 {'doi': '10.5286/isis.e.rb1790346-2',
  'enriched_subjects': {'+',
   'Cd',
   'Construction Engineering and Architecture',
   'Engineering',
   'Engineering Sciences',
   'Materials',
   'PG002',
   'RAVIL',
   'RLI838',
   'RLI882',
   'RLI904',
   'RLI908',
   'STFC ISIS Neutron and Muon Data',
   'Sine2020',
   'Technique Development',
   'cell'}}]

In [48]:
print(len(record_dois))

100


In [49]:
succesfully_enriched = [el for el in openaire_enriched if el['enriched_subjects']]
print(len(succesfully_enriched))

24


In [50]:
print(len(succesfully_enriched) / len(record_dois) * 100)

24.0


In [51]:
initial_set = set([clean_doi(doi) for doi in record_dois])
enriched_set = set([clean_doi(el['doi']) for el in openaire_enriched])

In [52]:
initial_set - enriched_set

set()

In [53]:
enriched_set - initial_set

set()

If both sets are empty, no DOIs have been added during enrichment and all initial DOIs have been preserved even with no subjects.